In [2]:
import pandas as pd
import numpy as np
from IPython.display import display
import matplotlib.pyplot as plt
import seaborn as sns
import decimal as dec
import re
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
movies = pd.read_csv('datasets/IMDb movies.csv', sep=";", encoding="latin1")
moviesaux = pd.read_csv('datasets/IMDb movies.csv', sep=";", encoding="latin1")
ratings = pd.read_csv('datasets/IMDb ratings.csv', sep=";")
ratingsaux = pd.read_csv('datasets/IMDb ratings.csv', sep=";")


In [4]:
#Parte da remoção de colunas do Pedro

missing_values_m = moviesaux.isnull().sum()
# print('MOVEIS', missing_values_m)

tamanho = len(movies.index)

moviesaux = moviesaux.drop(['imdb_title_id'], axis=1)

missing_values = movies.isnull().sum()

percentagem = (missing_values / tamanho) * 100

df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentagem': percentagem.round(2)
})

#print(df.to_string())
indice = df[df['Percentagem']> 50 ].index
movies = movies.drop(columns=indice)

#---------------------------------------------------------------

tamanho = len(ratings.index)

ratingsaux = ratingsaux.drop(['imdb_title_id'], axis=1)

missing_values = ratings.isnull().sum()

percentagem = (missing_values / tamanho) * 100

df = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentagem': percentagem.round(2)
})

indice = df[df['Percentagem']> 50 ].index
df.drop(indice, inplace=True) 

#print(df.to_string())

dados = ratings[~ratings.isna()]
colunas = [
    "allgenders_0age_votes", "allgenders_18age_votes",
    "allgenders_30age_votes", "allgenders_45age_votes",
    "males_allages_votes", "males_0age_votes", "males_18age_votes",
    "males_30age_votes", "males_45age_votes",
    "females_allages_votes", "females_0age_votes", "females_18age_votes",
    "females_30age_votes", "females_45age_votes",
    "top1000_voters_votes", "us_voters_votes", "non_us_voters_votes"
]

votos_total = dados['total_votes'].sum()
colunas_a_remover = []

for coluna in colunas:
    votos_demografia = dados[coluna].sum()
    proporcao =  votos_demografia / votos_total
    #print('Proporção de votos em percentagem da coluna', coluna, proporcao.round(2)*100)
    if proporcao < .10:
        colunas_a_remover.append(coluna)
    proporcao = 0

ratings = ratings.drop(columns=indice)

In [5]:
dataset = pd.merge(movies, ratings, on="imdb_title_id", how="inner")
pd.set_option('display.max_columns', None)  #Mostra todas as colunas
#display(dataset.head(5))

In [6]:
#---------------------------------------------------------------------------------------------------------------------------------------
#CRIAÇÃO DE UTILIZADORES
#---------------------------------------------------------------------------------------------------------------------------------------

users = list(range(1, 11)) #Criar 10 users
user_ratings = []
num_filmesfixos = 17
num_filmestotais = 25

#Escolher alguns filmes, para que haja garantia que temos mais do que uma review em maior parte dos filme.
#Desta forma reduz-se o número de zeros na matriz de similariedade
filmes_fixos = np.random.choice(dataset['imdb_title_id'], size = num_filmesfixos, replace = False)

for user in users:
    filmes_avaliados = list(filmes_fixos)

    #Preencher o resto com filmes aleatórios
    filmes_restantes = np.random.choice(
        dataset[~dataset['imdb_title_id'].isin(filmes_fixos)]['imdb_title_id'],
        size = num_filmestotais - num_filmesfixos, 
        replace = False
    )

    filmes_avaliados.extend(filmes_restantes)

    #Gerar ratings para cada filme avaliado pelo utilizador
    for movie_id in filmes_avaliados:
        avg_vote = dataset.loc[dataset['imdb_title_id'] == movie_id, 'avg_vote'].values[0]
        rating = np.clip(np.random.normal(avg_vote, 3), 1, 10) #Centro da distribuição é o avg_vote e a standard deviation é 2
        user_ratings.append({' User ID ': user, ' Movie ID ': movie_id, ' User Rating ': round(rating, 1)})


ratings_df = pd.DataFrame(user_ratings)
ratings_df.to_csv("user_ratings.csv", index=False) 
#print(ratings_df)



In [ ]:
#---------------------------------------------------------------------------------------------------------------------------------------
#ITEM-BASED COLLABORATIVE FILTERING
#---------------------------------------------------------------------------------------------------------------------------------------

user_ratings = pd.read_csv("user_ratings.csv") 

#Criar matriz utilizador-filme
user_movie_matrix = user_ratings.pivot(index = ' User ID ', columns = ' Movie ID ', values = ' User Rating ')
user_movie_matrix = user_movie_matrix.fillna(0)
#print(user_movie_matrix)

#Calcular similaridade entre filmes através de Cosine Similarity
item_similarity = cosine_similarity(user_movie_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity, index=user_movie_matrix.columns, columns=user_movie_matrix.columns)
#print(item_similarity_df.head())

#Função auxiliar para conseguir o nome do filme
def get_titulo(movie_id):
    title = dataset.loc[dataset['imdb_title_id'] == movie_id, 'title']
    return title.values[0]

#Função para recomendar filmes (IBC - Item-Based Collaborative)
def recomendar_IBC(movie_id, num_rec=5):
    if movie_id not in item_similarity_df.index:
        print("Filme não encontrado na base de dados!")
        return []
    
    similares_item = item_similarity_df[movie_id].sort_values(ascending=False)[1:num_rec+1]  
    recomendacoes_item = [get_titulo(filme) for filme in similares_item.index]  
    return recomendacoes_item

#Teste de recomendações
filme_teste = user_ratings[' Movie ID '].sample(1).values[0]
nome= get_titulo(filme_teste)
print(f"Recomendações para quem viu '{nome}': {recomendar_IBC(filme_teste)}")

#Ainda falta melhorar o cálculo das recomendações, apenas procura filmes com ratings parecidos, as recomendações não são as melhores
#Ás vezes aparece o nome do próprio filme nas recomendações, também ver isso


Recomendações para quem viu 'Una storia del West': ['Dummy', 'Cover Up', 'Il generale non si arrende', 'Kaithi', "L'adultero"]


In [8]:
#---------------------------------------------------------------------------------------------------------------------------------------
#USER-BASED COLLABORATIVE FILTERING
#---------------------------------------------------------------------------------------------------------------------------------------

user_ratings = pd.read_csv("user_ratings.csv") 

user_movie_matrix = user_ratings.pivot(index = ' User ID ', columns = ' Movie ID ', values = ' User Rating ')
user_movie_matrix = user_movie_matrix.fillna(0)
#print(user_movie_matrix)

#Calcular similaridade entre filmes através de Cosine Similarity
user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)
#print(user_similarity_df.head())


#Função para recomendar filmes (UBC - User-Based Collaborative)
def recomendar_UBC(user_id, num_rec = 3):

    similares_users = user_similarity_df[user_id].sort_values(ascending = False).index.tolist()
    similares_user_ratings = user_movie_matrix.loc[similares_users]
    recomendacoes = similares_user_ratings.mean(axis = 0)

    #Garantir que não temos filmes que o utilizador já viu nas recomendações
    user_rated_movies = user_movie_matrix.loc[user_id] 
    recomendacoes = recomendacoes[user_rated_movies == 0].sort_values(ascending = False)[0:5].index.tolist()
    recommended_movie_names = [get_titulo(movie_id) for movie_id in recomendacoes]

    return recommended_movie_names

# Testar as recomendações
utilizador = np.random.randint(1, 10) 
print(f"Recomendações para o Utilizador '{utilizador}':", recomendar_UBC(utilizador))

#Este filetring parece dar melhores resultados do que o anterior, benificia mais do cosine similarity

Recomendações para o Utilizador '8': ['Glass', 'Il nemico alle porte', 'La cittadella degli eroi', 'Je suis un assassin', 'Last Ferry']
